# PennyLane MPS convergence evidence

Compare a long-range 10-wire QNode with default.qubit and inspect MettleQ Dmax convergence metadata.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

The MPS device trades dense memory for bounded bond dimension and may truncate entanglement.

In [2]:
rng = np.random.default_rng(52)
angles = rng.uniform(-0.8, 0.8, size=(3, 10))
pairs = [[tuple(map(int, pair)) for pair in rng.permutation(10).reshape(5, 2)] for _ in range(3)]

def make_qnode(device):
    @qml.qnode(device)
    def circuit():
        for layer in range(3):
            for wire in range(10):
                qml.RY(angles[layer, wire], wires=wire)
            for first, second in pairs[layer]:
                qml.IsingZZ(0.43, wires=[first, second])
        return qml.state(), qml.expval(qml.Z(0))
    return circuit

reference_qnode = make_qnode(qml.device("default.qubit", wires=10))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(reference_qnode)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(
    wires=10,
    method="matrix_product_state",
    device="cpu",
    mps_max_bond_dimension=32,
    mps_truncation_threshold=1e-12,
    mps_convergence_bond_dimensions=(8, 16, 32),
    mps_convergence_atol=5e-4,
)
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(mettleq_qnode)
state_error = phase_aligned_statevector_error(reference[0], candidate[0])
expectation_error = abs(float(reference[1]) - float(candidate[1]))
convergence = mettleq_device.last_mps_convergence_report
accuracy = mettleq_device.last_mps_accuracy_report
convergence_summary = {
    "converged": convergence["converged"],
    "atol": convergence["atol"],
    "comparisons": convergence["comparisons"],
    "runs": [
        {
            "dmax": run["dmax"],
            "accuracy_classification": run["accuracy"]["classification"],
            "maximum_bond_dimension_reached": run["diagnostics"]["maximum_bond_dimension_reached"],
        }
        for run in convergence["runs"]
    ],
}
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

The exact state and expectation are checked, local telemetry must pass, and Dmax 16-to-32 agreement must meet tolerance.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/12_mps_convergence.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="MPS state atol=8e-5 and Dmax convergence atol=5e-4",
    passed=state_error <= 8e-5 and expectation_error <= 5e-5 and convergence["converged"] and accuracy["passed"],
    exact_match=bool(np.array_equal(reference[0], candidate[0])),
    selected_method=method,
    selected_device=device,
    metrics={"max_amplitude_error": state_error, "expectation_error": expectation_error, "accuracy": accuracy, "convergence": convergence_summary},
)


Comparison summary
------------------
Correctness contract: PASS — MPS state atol=8e-5 and Dmax convergence atol=5e-4
SDK reference median: 4.558 ms
MettleQ median:       47.129 ms
Timing interpretation: the SDK reference was 10.339x faster in this run.
MettleQ selected: matrix_product_state / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "MPS state atol=8e-5 and Dmax convergence atol=5e-4", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"accuracy": {"classification": "within_configured_local_thresholds", "observed": {"maximum_bond_dimension_reached": 32, "relative_discarded_weight_max": 8.770162032811263e-25, "relative_discarded_weight_sum": 1.6871274141988296e-24, "state_norm_error": 1.03001982498796e-08, "truncated": true}, "passed": true, "policy": "report", "schema_version": 1, "scope_warning": "Passing local telemetry thresholds 

## What should you conclude?

Use MPS for wide, structured circuits only after inspecting convergence; it is not automatically faster for small dense examples.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.